In [2]:
## IMPORT LIBRARIES ----------------------------------------------------------------------------------------------------
#https://github.com/nnc-ufmg/circadipy/blob/main/src/circadipy/analysis_examples/intellicage/intellicage_analysis.ipynb
%matplotlib qt

%reload_ext autoreload
%autoreload 3

from ranking_methods import rank_accuracy
from circadipy import chrono_reader as chr  
import sys                                                                                                              # Import sys to add paths to libraries                                                                                                           # Import re to work with regular expressions
import glob                                                                                                             # Import glob to read files                                                                                                   # Import numpy to work with arrays and make calculations                                                                                            # Import time to measure time
import os                                                                                                               # Import path to work with paths                
import pandas as pd
from ranking_methods import build_all_proxies, evaluate_proxies, rank_accuracy
import numpy as np
from scipy import stats
from datetime import date, datetime, timedelta
from itertools import combinations
from scipy.stats import rankdata, spearmanr, kendalltau
import json
                                                                                      # Import pandas to work with dataframes
import warnings                                                                                                         # Import warnings to ignore warnings
warnings.filterwarnings('ignore')                                                                                       # Ignore warnings

## IMPORT CIRCADIPY ----------------------------------------------------------------------------------------------------

parent_path = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.append(parent_path)

## PCA Visualization - Dimensionality Reduction
from models import train_rf, train_gb, train_ridge, train_adaboost, train_extratrees, train_logistic
from summary import generate_summary_report
from util import  get_data_scaled, generate_features, generate_temporal_features, calculate_correlations, build_animal_protocols, get_sorted_animals_files, combine_all_features
from analysis_visualization import generate_methods_comparison, plot_animals_activity, plot_correlation, plot_pca_analysis, plot_cross_correlation, summary_visualization

In [3]:
best_feat = 'ibi_median'
data_folder = "./data/dados_2026_06_08"

dataFiles = glob.glob(f'{data_folder}/*.zip')
output_folder = './results/predict_to_predict2'
ground_file = f'{data_folder}/ground.json'
os.makedirs(output_folder, exist_ok=True)
print(dataFiles)

if os.path.exists(ground_file):
    with open(ground_file, 'r') as f:
        ranks_ground = json.load(f)
        #ranks_ground = [int(f.split("_")[1]) for f in list(ranks_ground.keys())]

for k, v in ranks_ground.items():
    print(f"{k}: {v}")
root_folder = f"{data_folder}"    



['./data/dados_2026_06_08/2026-05-14 17.07.20.zip', './data/dados_2026_06_08/2026-05-18 12.26.42.zip', './data/dados_2026_06_08/2026-05-18 11.29.53.zip', './data/dados_2026_06_08/2026-05-15 16.33.28.zip', './data/dados_2026_06_08/2026-05-19 12.10.38.zip', './data/dados_2026_06_08/2026-05-12 11.46.29.zip', './data/dados_2026_06_08/2026-05-07 14.29.23.zip', './data/dados_2026_06_08/2026-05-08 14.28.22.zip', './data/dados_2026_06_08/2026-05-11 11.49.16.zip', './data/dados_2026_06_08/2026-05-22 18.27.08.zip', './data/dados_2026_06_08/2026-05-20 15.08.27.zip']
animal_6: 1
animal_1: 2
animal_11: 3
animal_10: 4
animal_12: 5
animal_8: 6
animal_5: 7
animal_9: 8
animal_2: 9
animal_4: 10
animal_7: 11
animal_3: 12


In [4]:

for file in dataFiles:
    zip_folder = file.split("/")[-1]
    sub_folder = zip_folder.split(".")[0].replace(" ", "_")
    os.makedirs(os.path.join(root_folder, sub_folder), exist_ok=True)
    a = chr.intellicage_unwrapper([file], sub_folder, sampling_interval = '30T')



File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_1.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_10.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_11.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_12.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_2.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_3.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_4.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_5.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_6.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_7.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_8.txt
File saved in ./data/dados_2026_06_08/2026-05-14_17/animal_9.txt
File saved in ./data/dados_2026_06_08/2026-05-18_12/animal_1.txt
File saved in ./data/dados_2026_06_08/2026-05-18_12/animal_10.txt
File saved in ./data/dados_2026_06_08/2026-05-18_12/animal_11.txt
File saved in ./data

In [5]:
individual_files = glob.glob(root_folder + "/**/*.txt", recursive=True)
individual_files = [f for f in individual_files if "animal_" in f]
apply_filtering = True
animals = [int(k.split("_")[1]) for k in ranks_ground.keys()]
animals_files = get_sorted_animals_files(individual_files, animals)
animals_protocols, animals_by_day = build_animal_protocols(animals_files, apply_filtering=apply_filtering)

Animal 1: 10
Animal 2: 11
Animal 3: 10
Animal 4: 11
Animal 5: 10
Animal 6: 11
Animal 7: 10
Animal 8: 11
Animal 9: 10
Animal 10: 10
Animal 11: 11
Animal 12: 10
Animal2 1
savgol True
Animal2 2
savgol True
Animal2 3
savgol True
Animal2 4
savgol True
Animal2 5
savgol True
Animal2 6
savgol True
Animal2 7
savgol True
Animal2 8
savgol True
Animal2 9
savgol True
Animal2 10
savgol True
Animal2 11
savgol True
Animal2 12
savgol True
Animal animal_1 Days found: 22
Animal animal_2 Days found: 22
Animal animal_3 Days found: 22
Animal animal_4 Days found: 22
Animal animal_5 Days found: 22
Animal animal_6 Days found: 22
Animal animal_7 Days found: 22
Animal animal_8 Days found: 22
Animal animal_9 Days found: 22
Animal animal_10 Days found: 22
Animal animal_11 Days found: 22
Animal animal_12 Days found: 22


In [ ]:
output_path = f'{output_folder}/basic_features.csv'
features_df = generate_features(animals_protocols, output_path=output_path)

output_path = f'{output_folder}/temporal_features.csv'
temporal_df = generate_temporal_features(animals_protocols, output_path=output_path)

output_path = f'{output_folder}/all_features.csv'
all_features, feature_cols = combine_all_features(features_df, temporal_df, output_path=output_path)

X_scaled =  get_data_scaled(all_features, feature_cols)
y = []
for animal in all_features['animal'].tolist():
    y.append(ranks_ground[animal])



for animal, rank in sorted(ranks_ground.items(), key=lambda x: x[1]):
    print(f"{rank}: {animal}")

all_features['actual_rank'] = y
all_features = all_features.sort_values('actual_rank')   # reassign to keep the sorted version
# [9, 8, 11, 10, 12, 1, 5, 6, 7, 2, 3, 4]

all_features.head()

Saving features on ./results/predict_to_predict2/basic_features.csv
Saving temporal features on ./results/predict_to_predict2/temporal_features.csv
Saving all features on ./results/predict_to_predict2/all_features.csv
1: animal_6
2: animal_1
3: animal_11
4: animal_10
5: animal_12
6: animal_8
7: animal_5
8: animal_9
9: animal_2
10: animal_4
11: animal_7
12: animal_3


,animal,total_activity,mean_activity,std_activity,max_activity,min_activity,cv_activity,median_activity,max_median_ratio,activity_per_hour,...,night_bout_len_mean,night_bout_len_cv,night_transitions_per_hour,night_onset_latency_h,night_first_2h_frac,night_gini,night_peak_hour,night_activity_per_bout,night_day_intensity_ratio,actual_rank
animal_6,animal_6,1256.0,0.623325,0.903556,5.142857,-0.942857,1.449575,0.314286,16.363636,52.333333,...,0.440559,0.650503,1.136048,4.75,-0.030748,3.644838,1.0,0.938378,0.141789,1
animal_1,animal_1,2180.0,1.081886,1.849510,14.057143,-2.000000,1.709524,0.342857,41.000000,90.833333,...,0.605769,0.767178,0.826216,4.00,0.000000,0.000000,0.0,-1.699829,-0.083451,2
animal_11,animal_11,2157.0,1.070471,1.555417,11.657143,-2.057143,1.453020,0.342857,34.000000,89.875000,...,0.516393,0.658156,0.969215,4.75,-0.030455,3.624485,0.0,1.051214,0.076076,3
animal_10,animal_10,2081.0,1.032754,1.739299,10.971429,-1.771429,1.684137,0.342857,32.000000,86.708333,...,0.649485,0.817306,0.770606,4.75,0.000000,0.000000,1.0,-0.869460,-0.051572,4
animal_12,animal_12,2164.0,1.073945,1.634248,10.285714,-1.028571,1.521723,0.342857,30.000000,90.166667,...,0.538462,0.666667,0.929494,4.75,-0.075096,8.201724,0.0,0.424263,0.029404,5


In [24]:


feature_rhos_path = "./data/dados_iniciais_estruturados/feature_rhos.csv"
feature_rhos_df = pd.read_csv(feature_rhos_path)

if {'feature', 'rho'}.issubset(feature_rhos_df.columns):
    feature_rhos = feature_rhos_df.set_index('feature')['rho']
else:
    raise ValueError(
        "`feature_rhos_previous.csv` must contain `feature` and `rho` columns. "
        "Recreate it with: feature_rhos.to_frame('rho').rename_axis('feature').to_csv(...)"
    )

feature_rhos.head()

feature
rhythm_ratio_24_over_harm    0.790210
power_24h                    0.706294
autocorr_12h                -0.664336
power_12h                   -0.657343
cosinor_amplitude            0.657343
Name: rho, dtype: float64

In [25]:

ys = [int(i.split("_")[1]) for i in all_features['animal'].tolist()]
best_feat = 'cosinor_amplitude'
combos = [['cosinor_amplitude'], ['power_24h', 'high_activity_frac', 'activity_per_bout']]



result = {}
cont = 0
for named_combo in combos:

    feature_rhos, proxies_raw = build_all_proxies(
        all_features, feature_cols, X_scaled, None,
        k=3, best_feat_idx=best_feat,
        named_combo=named_combo,
        feature_rhos=feature_rhos
    )
    for k, v in proxies_raw.items():
        print(f"{k}: {v}")



    best_feature_key = f'Best feature ({best_feat})'

    scores = proxies_raw[best_feature_key]
    pred_rank = rankdata(scores, method='ordinal')
    #print(pred_rank)


    output_best_feature = f'{data_folder}/pred_{best_feat}.csv'

    best_combo_key = f'Best combo ({ " + ".join(named_combo) })'
    scores = proxies_raw[best_combo_key]
    pred_rank = rankdata(scores, method='ordinal')


    combo_name = "_".join(named_combo)

    if combo_name not in result:
        summary = {"pred": pred_rank, "ground": ys}
        metrics = rank_accuracy(y, pred_rank)

        summary.update(metrics)

        result[combo_name] = summary


df = pd.DataFrame(result)
df.to_csv(f"./pred_data_to_predict_08.csv", index=False)


print(result)


Using feature_rhos provided externally (e.g. from a reference dataset).
Best feature (cosinor_amplitude): [0.3495697  0.31492114 0.34955908 0.26930195 0.25913889 0.25314785
 0.40910995 0.35359512 0.23325785 0.45126302 0.29124483 0.41784596]
Best combo (cosinor_amplitude): [-0.21149127 -1.41016598  1.29926566  1.78976977  1.17103626  0.29708926
 -0.55901824 -1.11821536  0.35617534 -0.88110133  0.29693328 -1.0302774 ]
Combo sign-aligned mean (cosinor_amplitude): [-0.21149127 -1.41016598  1.29926566  1.78976977  1.17103626  0.29708926
 -0.55901824 -1.11821536  0.35617534 -0.88110133  0.29693328 -1.0302774 ]
Using feature_rhos provided externally (e.g. from a reference dataset).
Best feature (cosinor_amplitude): [0.3495697  0.31492114 0.34955908 0.26930195 0.25913889 0.25314785
 0.40910995 0.35359512 0.23325785 0.45126302 0.29124483 0.41784596]
Best combo (power_24h + high_activity_frac + activity_per_bout): [-1.97743454 -0.64009066  4.30367554  1.54514224 -0.70545626  1.45826443
 -0.99261